In [31]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.svm import SVC
# StratifiedKFold는 일반적인 KFold 교차 검증과는 다르게 라벨링 비율을 유지하면서 데이터를 추출한다.
from sklearn.model_selection import StratifiedKFold # 교차 검증 알고리즘을 사용하기 위해 import 한다.
# 그리드 서치는 사용자가 지정한 몇 가지 잠재적인 하이퍼파라미터 후보군들의 조합 중에서 가장 좋은 조합을 찾아준다.
# 그리드 서치는 사용자가 하이퍼파라미터 후보군들을 하나하나 대입하면서 오차를 확인하는 작업을 그리드 서치가 대신해서 손쉽게 사용 할 수 있다.
from sklearn.model_selection import GridSearchCV # 그리드 서치 알고리즘을 사용하기 위해 import 한다.
from sklearn.model_selection import cross_validate # 교차 검증 스코어를 확인하기 위해 import 한다.
from sklearn.model_selection import cross_val_score # 교차 검증 스코어를 확인하기 위해 import 한다.

교차 검증(cross validation)

앞선 지도 학습 알고리즘에서는 전체 데이터를 학습 데이터와 테스트 데이터로 나눠 모델을 학습 시켰다.

<table align="left" width="700">
    <tr>
        <td colspan="6" style="border: 1px solid;">
            <div style="text-align: center">Total Data</div>
        </td>
    </tr>
    <tr>
        <td colspan="5" style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Test</div>
        </td>
    </tr>
    <tr>
        <td>
        </td>
    </tr>
    <tr>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Validation</div>
        </td>
        <td rowspan="5" style="border: 1px solid;">
            <div style="text-align: center">
                파라미터,<br>
                하이퍼파라미터<br>
                설정
            </div>
        </td>
    </tr>
    <tr>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Validation</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
    </tr>
    <tr>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Validation</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
    </tr>
    <tr>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Validation</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
    </tr>
    <tr>
        <td style="border: 1px solid;">
            <div style="text-align: center">Validation</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Train</div>
        </td>
    </tr>
    <tr>
        <td>
        </td>
    </tr>
    <tr>
        <td colspan="5">
            <div>모형평가</div>
        </td>
        <td style="border: 1px solid;">
            <div style="text-align: center">Test</div>
        </td>
    </tr>
</table>

와인 데이터를 사용해서 하이퍼파라미터 튜닝을 위해 교차 검증 기법을 활용해 와인 종류를 분류하기 위해 데이터를 불러오고 표준화 한다.

In [2]:
# 데이터 불러오기
raw_data = datasets.load_wine() # 사이킷런 라이브러리가 제공하는 와인 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

그리드 서치 모델을 생성하고 학습시켜서 최적의 하이퍼파라미터를 찾는다.

In [3]:
model = SVC(probability=True) # 서포트 벡터 머신 모델을 만든다.

# 그리드 서치에서 사용할 하이퍼파라미터 후보군을 설정한다.
param_grid = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'C': [0.1, 1, 10, 100]
}

# StratifiedKFold는 n_splits 속성값으로 학습 데이터를 나눌 개수를 지정하고, shuffle 속성값을 True를 지정해서 데이터를 섞이게 해서 교차 검증 객체를 만든다.
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# GridSearchCV는 estimator 속성값으로 그리드 서치를 적용할 모델, param_grid 속성값으로 하이퍼파라미터 후보군, cv 속성값으로 kfold 객체, scoring 속성값으로
# 모델 평가 방법을 지정해서 그리드 서치 객체를 만든다.
# grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=kfold, scoring='accuracy')
# grid.fit(x_train, y_train) # 표준화된 학습 데이터(x_train)와 학습 데이터에 따른 레이블(y_train)을 넘겨서 그리드 서치를 학습시킨다.
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=kfold, scoring='accuracy').fit(x_train, y_train)

그리드 서치 결과를 확인

In [4]:
# cv_results_ 속성으로 각각의 하이퍼파라미터 후보군 조합에 대한 그리드 서치 결과를 확인할 수 있다.
print(grid.cv_results_)

{'mean_fit_time': array([0.00290518, 0.00438213, 0.00237503, 0.00456638, 0.00176826,
       0.00247908, 0.00183315, 0.00175743, 0.00192528, 0.00230923,
       0.00182219, 0.00155935, 0.00186501, 0.00251408, 0.00201778,
       0.00171738]), 'std_fit_time': array([0.00185598, 0.00122475, 0.00044227, 0.00162776, 0.0004837 ,
       0.00028329, 0.00069306, 0.00064301, 0.0006995 , 0.0004019 ,
       0.00050105, 0.00049612, 0.0003679 , 0.00057363, 0.00031161,
       0.00074949]), 'mean_score_time': array([0.00103254, 0.00131326, 0.00091815, 0.00119724, 0.00101614,
       0.00081811, 0.00070887, 0.00061145, 0.00130544, 0.00061173,
       0.00076818, 0.00101743, 0.00051184, 0.0005218 , 0.0006135 ,
       0.00019956]), 'std_score_time': array([8.49741495e-04, 4.12050972e-04, 4.96204935e-04, 3.91066087e-04,
       2.79261342e-05, 4.10293939e-04, 6.05188929e-04, 5.83797316e-04,
       3.73508972e-04, 4.99989180e-04, 6.39306250e-04, 5.38487531e-04,
       4.60272644e-04, 6.52813573e-04, 5.01696923e

In [5]:
# cv_results_ 속성값을 텍스트로 확인하면 보기 불편하므로 데이터프레임으로 보기 편하게 출력한다.
pd.DataFrame(grid.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002905,0.001856,0.001033,0.000850,0.1,linear,"{'C': 0.1, 'kernel': 'linear'}",0.962963,0.962963,0.925926,1.000000,0.923077,0.954986,0.028341,9
1,0.004382,0.001225,0.001313,0.000412,0.1,rbf,"{'C': 0.1, 'kernel': 'rbf'}",0.925926,1.000000,0.962963,0.961538,1.000000,0.970085,0.027798,1
2,0.002375,0.000442,0.000918,0.000496,0.1,poly,"{'C': 0.1, 'kernel': 'poly'}",0.666667,0.740741,0.814815,0.615385,0.730769,0.713675,0.068007,16
3,0.004566,0.001628,0.001197,0.000391,0.1,sigmoid,"{'C': 0.1, 'kernel': 'sigmoid'}",0.962963,1.000000,0.962963,1.000000,0.923077,0.969801,0.028638,2
4,0.001768,0.000484,0.001016,0.000028,1.0,linear,"{'C': 1, 'kernel': 'linear'}",0.888889,0.962963,0.925926,1.000000,0.846154,0.924786,0.054014,12
5,0.002479,0.000283,0.000818,0.000410,1.0,rbf,"{'C': 1, 'kernel': 'rbf'}",0.925926,0.962963,0.962963,0.961538,1.000000,0.962678,0.023431,3
6,0.001833,0.000693,0.000709,0.000605,1.0,poly,"{'C': 1, 'kernel': 'poly'}",0.888889,0.962963,1.000000,0.961538,0.884615,0.939601,0.045322,10
7,0.001757,0.000643,0.000611,0.000584,1.0,sigmoid,"{'C': 1, 'kernel': 'sigmoid'}",0.925926,0.962963,0.925926,1.000000,0.961538,0.955271,0.027646,7
8,0.001925,0.000699,0.001305,0.000374,10.0,linear,"{'C': 10, 'kernel': 'linear'}",0.888889,0.962963,0.925926,1.000000,0.846154,0.924786,0.054014,12
9,0.002309,0.000402,0.000612,0.000500,10.0,rbf,"{'C': 10, 'kernel': 'rbf'}",0.925926,0.962963,0.962963,0.961538,1.000000,0.962678,0.023431,3


In [7]:
# 모니터 해상도가 작아서 보기 불편하면 넘파이의 transpose() 메소드를 실행해서 전치시켜 확인하면 된다.
np.transpose(pd.DataFrame(grid.cv_results_))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
mean_fit_time,0.002905,0.004382,0.002375,0.004566,0.001768,0.002479,0.001833,0.001757,0.001925,0.002309,0.001822,0.001559,0.001865,0.002514,0.002018,0.001717
std_fit_time,0.001856,0.001225,0.000442,0.001628,0.000484,0.000283,0.000693,0.000643,0.000699,0.000402,0.000501,0.000496,0.000368,0.000574,0.000312,0.000749
mean_score_time,0.001033,0.001313,0.000918,0.001197,0.001016,0.000818,0.000709,0.000611,0.001305,0.000612,0.000768,0.001017,0.000512,0.000522,0.000613,0.0002
std_score_time,0.00085,0.000412,0.000496,0.000391,0.000028,0.00041,0.000605,0.000584,0.000374,0.0005,0.000639,0.000538,0.00046,0.000653,0.000502,0.000399
param_C,0.1,0.1,0.1,0.1,1.0,1.0,1.0,1.0,10.0,10.0,10.0,10.0,100.0,100.0,100.0,100.0
param_kernel,linear,rbf,poly,sigmoid,linear,rbf,poly,sigmoid,linear,rbf,poly,sigmoid,linear,rbf,poly,sigmoid
params,"{'C': 0.1, 'kernel': 'linear'}","{'C': 0.1, 'kernel': 'rbf'}","{'C': 0.1, 'kernel': 'poly'}","{'C': 0.1, 'kernel': 'sigmoid'}","{'C': 1, 'kernel': 'linear'}","{'C': 1, 'kernel': 'rbf'}","{'C': 1, 'kernel': 'poly'}","{'C': 1, 'kernel': 'sigmoid'}","{'C': 10, 'kernel': 'linear'}","{'C': 10, 'kernel': 'rbf'}","{'C': 10, 'kernel': 'poly'}","{'C': 10, 'kernel': 'sigmoid'}","{'C': 100, 'kernel': 'linear'}","{'C': 100, 'kernel': 'rbf'}","{'C': 100, 'kernel': 'poly'}","{'C': 100, 'kernel': 'sigmoid'}"
split0_test_score,0.962963,0.925926,0.666667,0.962963,0.888889,0.925926,0.888889,0.925926,0.888889,0.925926,0.925926,0.888889,0.888889,0.925926,0.888889,0.814815
split1_test_score,0.962963,1.0,0.740741,1.0,0.962963,0.962963,0.962963,0.962963,0.962963,0.962963,1.0,0.962963,0.962963,0.962963,1.0,0.962963
split2_test_score,0.925926,0.962963,0.814815,0.962963,0.925926,0.962963,1.0,0.925926,0.925926,0.962963,0.925926,0.851852,0.925926,0.962963,0.962963,0.888889


best 스코어와 best 스코어를 낸 하이퍼파라미터를 확인한다.

In [10]:
# best_index_ 속성으로 그리드 서치가 beat 스코어를 낸 하이퍼파라미터의 인덱스를 얻어오다.
print(grid.best_index_)
# best_params_ 속성으로 그리드 서치가 beat 스코어를 낸 하이퍼파라미터를 얻어오다.
print(grid.best_params_)
# best_score_ 속성으로 그리드 서치가 beat 스코어를 얻어오다.
print(grid.best_score_)
# best_estimator_ 속성으로 그리드 서치가 beat 스코어를 낸 하이퍼파라미터가 적용된 모델을 얻어오다.
print(grid.best_estimator_)

1
{'C': 0.1, 'kernel': 'rbf'}
0.9700854700854702
SVC(C=0.1, probability=True)


best 스코어를 낸 하이퍼파라미터를 적용한 모델을 최종 모델로 선정한다.

In [11]:
model = grid.best_estimator_

그리드 서치 결과에서 best 스코어를 낸 모델에서 교차 검증 스코어를 확인한다.

In [30]:
scoring=['accuracy', 'precision_macro','recall_macro', 'f1_macro']
# scoring 속성값에 지정할 수 있는 평가 지표는 'accuracy', 'precision_macro','recall_macro', 'f1_macro' 중 하나 이상을 지정하면 된다.
cv_score = cross_validate(estimator=model, X=x_train, y=y_train, cv=kfold, scoring=scoring)

for key, value in cv_score.items():
    print(key, value)

fit_time [0.0076201  0.0065186  0.00300193 0.00350952 0.00314307]
score_time [0.01003957 0.00505161 0.00564766 0.00558209 0.00476336]
test_accuracy [0.92592593 1.         0.96296296 0.96153846 1.        ]
test_precision_macro [0.925      1.         0.96969697 0.96969697 1.        ]
test_recall_macro [0.925      1.         0.96296296 0.95833333 1.        ]
test_f1_macro [0.925      1.         0.96451914 0.96190476 1.        ]


In [38]:
# cross_validate() 함수와 cross_val_score() 함수는 모두 교차 검증 스코어를 출력한다.
# cross_validate() 함수와 cross_val_score() 함수의 차이점은 scoring 속성값으로 cross_validate() 함수는 딱 여러개를 지정할 수 있고 cross_val_score() 함수는
# 딱 한 개만 지정할 수 있다.
cv_score = cross_val_score(estimator=model, X=x_train, y=y_train, cv=kfold, scoring='accuracy')
# cv_score = cross_val_score(estimator=model, X=x_train, y=y_train, cv=kfold, scoring='precision_macro')
# cv_score = cross_val_score(estimator=model, X=x_train, y=y_train, cv=kfold, scoring='recall_macro')
# cv_score = cross_val_score(estimator=model, X=x_train, y=y_train, cv=kfold, scoring='f1_macro')
print(cv_score)

[0.92592593 1.         0.96296296 0.96153846 1.        ]


학습된 모델로 테스트 데이터를 예측한다.

In [40]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 서포트 벡터 머신 모델을 예측한다.
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 1 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [41]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)

[[9.92352969e-01 2.65309297e-03 4.99393806e-03]
 [1.14807777e-02 1.20177498e-02 9.76501473e-01]
 [5.48035090e-02 9.38308771e-01 6.88771993e-03]
 [9.88400585e-01 5.03903510e-03 6.56037945e-03]
 [4.40500472e-02 8.94782306e-01 6.11676465e-02]
 [8.11505261e-02 8.22620247e-01 9.62292266e-02]
 [9.89469696e-01 3.24545170e-03 7.28485196e-03]
 [5.02703421e-03 5.25866805e-03 9.89714298e-01]
 [2.62624937e-03 9.95030687e-01 2.34306336e-03]
 [2.32570637e-03 9.86330107e-01 1.13441862e-02]
 [1.41715746e-02 2.83222349e-02 9.57506191e-01]
 [1.94639563e-02 2.94956860e-02 9.51040358e-01]
 [9.95088891e-01 7.64444650e-04 4.14666444e-03]
 [2.27000014e-01 7.59067604e-01 1.39323822e-02]
 [7.21212230e-03 4.34305301e-03 9.88444825e-01]
 [2.07901390e-03 9.96575557e-01 1.34542958e-03]
 [9.12130195e-01 5.94606564e-02 2.84091485e-02]
 [9.73719429e-01 8.96387190e-03 1.73166994e-02]
 [2.30254385e-02 6.13365327e-01 3.63609235e-01]
 [9.93903471e-01 2.21189925e-03 3.88462933e-03]
 [1.37433645e-01 8.40918009e-01 2.164834

학습된 모델을 평가한다.

In [42]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[16  0  0]
 [ 0 21  0]
 [ 0  0  8]]


In [43]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        16
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00         8

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [44]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

1.0


In [45]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[1. 1. 1.]


In [46]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[1. 1. 1.]


In [47]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[1. 1. 1.]
